In [ ]:
import json
import shutil
from collections import defaultdict
from pathlib import Path
import pandas as pd
from historical_ner_processing.digiknihovny_utils import retrieve_digiknihovny_resources

In [ ]:
JSONL_PATH = Path(
    "../datasets/peoplegators/people_gator__face_and_name__2026-01-27.jsonl"
)
DATA_ROOT = Path("../../mnt/")
EXPORT_ROOT = Path("../datasets/digiknihovna_data")

EXPORT_ALL = True
NUM_PAGES = 20

SHUFFLE = True
SEED = 42

OVERWRITE_EXISTING_PAGES = False
OVERWRITE_METADATA = False

FAST_SKIP_COMPLETED_DOCS = True

SAVE_DETECTION_METADATA = True
COPY_SELECTED_PAGES_TO_DETECTIONS = True
COPY_SELECTED_PAGES_TO_IMAGES = False

EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

print("JSONL_PATH =", JSONL_PATH)
print("DATA_ROOT  =", DATA_ROOT)
print("EXPORT_ROOT=", EXPORT_ROOT)
print("EXPORT_ALL =", EXPORT_ALL)
print("NUM_PAGES  =", NUM_PAGES)
print("OVERWRITE_EXISTING_PAGES =", OVERWRITE_EXISTING_PAGES)
print("OVERWRITE_METADATA       =", OVERWRITE_METADATA)
print("FAST_SKIP_COMPLETED_DOCS =", FAST_SKIP_COMPLETED_DOCS)
print("SAVE_DETECTION_METADATA  =", SAVE_DETECTION_METADATA)
print("COPY_SELECTED_PAGES_TO_DETECTIONS =", COPY_SELECTED_PAGES_TO_DETECTIONS)
print("COPY_SELECTED_PAGES_TO_IMAGES     =", COPY_SELECTED_PAGES_TO_IMAGES)

In [ ]:
def load_unique_pages_from_jsonl(jsonl_path: Path) -> pd.DataFrame:
    rows = []
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)

            di = obj.get("document_info", {}) if isinstance(obj, dict) else {}

            library = di.get(
                "library", obj.get("library") if isinstance(obj, dict) else None
            )
            document = di.get(
                "document", obj.get("document") if isinstance(obj, dict) else None
            )
            page = di.get("page", obj.get("page") if isinstance(obj, dict) else None)

            rows.append(
                {
                    "library": library,
                    "document": document,
                    "page": page,
                }
            )

    df = (
        pd.DataFrame(rows)
        .dropna(subset=["library", "document", "page"])
        .astype({"library": str, "document": str, "page": str})
        .drop_duplicates()
        .sort_values(["library", "document", "page"])
        .reset_index(drop=True)
    )
    return df


def select_pages(
    df: pd.DataFrame,
    export_all: bool,
    num_pages: int,
    shuffle: bool = True,
    seed: int = 42,
) -> pd.DataFrame:
    if export_all:
        out = df.copy()
        if shuffle:
            out = out.sample(frac=1, random_state=seed)
        return out.reset_index(drop=True)

    n = min(num_pages, len(df))
    if shuffle:
        return df.sample(n=n, random_state=seed).reset_index(drop=True)
    return df.head(n).copy().reset_index(drop=True)


pages_df = load_unique_pages_from_jsonl(JSONL_PATH)
selected_df = select_pages(pages_df, EXPORT_ALL, NUM_PAGES, shuffle=SHUFFLE, seed=SEED)

print(f"Počet unikátnych strán v JSONL: {len(pages_df)}")
print(f"Počet vybraných strán na export: {len(selected_df)}")
display(selected_df.head(20))

In [ ]:
def group_requested_pages(
    selected_pages_df: pd.DataFrame,
) -> dict[tuple[str, str], set[str]]:
    grouped = defaultdict(set)
    for _, row in selected_pages_df.iterrows():
        grouped[(row["library"], row["document"])].add(str(row["page"]))
    return dict(grouped)


requested_by_doc = group_requested_pages(selected_df)
print(
    f"Počet dokumentov, ktoré sa budú čítať cez historical_ner_processing: {len(requested_by_doc)}"
)
list(requested_by_doc.items())[:5]

In [ ]:
def record_to_jsonable(r):
    if hasattr(r, "model_dump"):
        return r.model_dump()
    if hasattr(r, "dict"):
        return r.dict()
    if isinstance(r, dict):
        return r
    return {"value": str(r)}


def write_jsonl(records, out_path: Path, overwrite: bool = False):
    if out_path.exists() and not overwrite:
        return "skipped_existing"

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for r in records or []:
            f.write(json.dumps(record_to_jsonable(r), ensure_ascii=False) + "\n")
    return "written"


def build_image_index(image_paths, page_names, images):
    index = {}

    def add_key(key, payload):
        if key is None:
            return
        key = str(key)
        if key not in index:
            index[key] = payload
        stem = Path(key).stem
        if stem and stem not in index:
            index[stem] = payload

    if image_paths:
        for p in image_paths:
            p = Path(p)
            payload = {
                "kind": "path",
                "src": p,
                "out_name": p.name,
            }
            add_key(p.name, payload)
            add_key(p.stem, payload)

    if page_names and images:
        for page_name, img in zip(page_names, images):
            page_name = str(page_name)
            payload = {
                "kind": "pil",
                "src": img,
                "out_name": (
                    Path(page_name).name if page_name else f"{len(index):04d}.jpg"
                ),
            }
            add_key(page_name, payload)
            add_key(Path(page_name).name, payload)
            add_key(Path(page_name).stem, payload)

    return index


def save_page_resource(resource: dict, dst: Path, overwrite: bool = False):
    if dst.exists() and not overwrite:
        return "skipped_existing"

    dst.parent.mkdir(parents=True, exist_ok=True)

    if resource["kind"] == "path":
        shutil.copy2(resource["src"], dst)
        return "copied"

    if resource["kind"] == "pil":
        img = resource["src"]
        suffix = dst.suffix.lower()
        if suffix in {".jpg", ".jpeg"}:
            img.save(dst, quality=95)
        else:
            img.save(dst)
        return "saved"

    raise ValueError(f"Neznámy typ resource: {resource['kind']}")


def page_candidate_names(page: str) -> list[str]:
    page = str(page)
    page_path = Path(page)
    candidates = []

    def add(name):
        if name and name not in candidates:
            candidates.append(name)

    add(page)
    add(page_path.name)

    stem = page_path.stem
    suffix = page_path.suffix.lower()

    if stem:
        add(stem)

    if not suffix and stem:
        for ext in [".jpg", ".jpeg", ".png", ".tif", ".tiff", ".webp"]:
            add(stem + ext)

    return candidates


def any_target_exists(base_dir: Path, page: str) -> tuple[bool, str | None]:
    if not base_dir.exists():
        return False, None

    for candidate in page_candidate_names(page):
        p = base_dir / candidate
        if p.exists():
            return True, candidate

    return False, None


def evaluate_document_precheck(
    base: Path,
    requested_pages: set[str],
    copy_to_detections: bool,
    copy_to_images: bool,
    save_detection_metadata: bool,
    overwrite_existing_pages: bool,
    overwrite_metadata: bool,
) -> dict:
    out_det = base / "detections"
    out_img = base / "images"

    page_checks = []
    all_pages_present = True

    for page in sorted(requested_pages):
        det_exists, det_name = (False, None)
        img_exists, img_name = (False, None)

        if copy_to_detections:
            det_exists, det_name = any_target_exists(out_det, page)

        if copy_to_images:
            img_exists, img_name = any_target_exists(out_img, page)

        page_complete = True
        if copy_to_detections and not det_exists:
            page_complete = False
        if copy_to_images and not img_exists:
            page_complete = False

        if overwrite_existing_pages:
            page_complete = False

        page_checks.append(
            {
                "page": str(page),
                "detections_exists": det_exists,
                "detections_name": det_name,
                "images_exists": img_exists,
                "images_name": img_name,
                "page_complete": page_complete,
            }
        )

        if not page_complete:
            all_pages_present = False

    labels_path = out_det / "peoplegator_labels.jsonl"
    ners_path = out_det / "peoplegator_ners.jsonl"

    metadata_complete = True
    if save_detection_metadata:
        metadata_complete = labels_path.exists() and ners_path.exists()
        if overwrite_metadata:
            metadata_complete = False

    skip_retrieve = all_pages_present and metadata_complete

    return {
        "skip_retrieve": skip_retrieve,
        "all_pages_present": all_pages_present,
        "metadata_complete": metadata_complete,
        "page_checks": page_checks,
        "labels_path": labels_path,
        "ners_path": ners_path,
    }

In [ ]:
processed_docs = []
copied_rows = []
skipped_rows = []
missing_rows = []
failed_docs = []

for (library, document), requested_pages in requested_by_doc.items():
    print(f"\n=== {library} / {document} ===")

    base = EXPORT_ROOT / library / document
    out_det = base / "detections"
    out_img = base / "images"

    precheck = evaluate_document_precheck(
        base=base,
        requested_pages=requested_pages,
        copy_to_detections=COPY_SELECTED_PAGES_TO_DETECTIONS,
        copy_to_images=COPY_SELECTED_PAGES_TO_IMAGES,
        save_detection_metadata=SAVE_DETECTION_METADATA,
        overwrite_existing_pages=OVERWRITE_EXISTING_PAGES,
        overwrite_metadata=OVERWRITE_METADATA,
    )

    if FAST_SKIP_COMPLETED_DOCS and precheck["skip_retrieve"]:
        for chk in precheck["page_checks"]:
            skipped_rows.append(
                {
                    "library": library,
                    "document": document,
                    "page": chk["page"],
                    "out_name": chk["detections_name"]
                    or chk["images_name"]
                    or chk["page"],
                    "source_kind": "precheck_existing",
                    "labels_status": (
                        "skipped_existing" if SAVE_DETECTION_METADATA else None
                    ),
                    "ners_status": (
                        "skipped_existing" if SAVE_DETECTION_METADATA else None
                    ),
                    "precheck_skipped_doc": True,
                }
            )

        processed_docs.append(
            {
                "library": library,
                "document": document,
                "requested_pages": len(requested_pages),
                "copied_pages": 0,
                "skipped_existing_pages": len(requested_pages),
                "missing_pages": 0,
                "face_detections": None,
                "ners": None,
                "labels_status": (
                    "skipped_existing" if SAVE_DETECTION_METADATA else None
                ),
                "ners_status": "skipped_existing" if SAVE_DETECTION_METADATA else None,
                "precheck_skipped_doc": True,
                "precheck_all_pages_present": precheck["all_pages_present"],
                "precheck_metadata_complete": precheck["metadata_complete"],
            }
        )

        print(
            "SKIP WITHOUT RETRIEVE:",
            f"requested={len(requested_pages)}",
            "all required pages and metadata already exist in export",
        )
        continue

    try:
        page_layouts, page_names, images, image_paths, face_detections, ners = (
            retrieve_digiknihovny_resources(
                DATA_ROOT,
                library,
                document,
                return_page_xmls=True,
                return_images=True,
                return_people_gator=True,
                return_people_gator_ners=True,
            )
        )

        out_det.mkdir(parents=True, exist_ok=True)
        if COPY_SELECTED_PAGES_TO_IMAGES:
            out_img.mkdir(parents=True, exist_ok=True)

        labels_status = None
        ners_status = None
        if SAVE_DETECTION_METADATA:
            labels_status = write_jsonl(
                face_detections or [],
                out_det / "peoplegator_labels.jsonl",
                overwrite=OVERWRITE_METADATA,
            )
            ners_status = write_jsonl(
                ners or [],
                out_det / "peoplegator_ners.jsonl",
                overwrite=OVERWRITE_METADATA,
            )

        image_index = build_image_index(image_paths, page_names, images)

        doc_copied = 0
        doc_skipped = 0
        doc_missing = 0

        for page in sorted(requested_pages):
            resource = image_index.get(str(page))
            if resource is None:
                missing_rows.append(
                    {
                        "library": library,
                        "document": document,
                        "page": page,
                        "reason": "page_not_found_in_retrieved_resources",
                    }
                )
                doc_missing += 1
                continue

            page_action = None
            saved_image_path = None

            if COPY_SELECTED_PAGES_TO_IMAGES:
                img_dst = base / "images" / resource["out_name"]
                page_action = save_page_resource(
                    resource, img_dst, overwrite=OVERWRITE_EXISTING_PAGES
                )
                saved_image_path = img_dst

            if COPY_SELECTED_PAGES_TO_DETECTIONS:
                det_dst = base / "detections" / resource["out_name"]

                # ak je stránka uložená v images/, do detections ju kopíruj odtiaľ
                if saved_image_path is not None and saved_image_path.exists():
                    if det_dst.exists() and not OVERWRITE_EXISTING_PAGES:
                        det_status = "skipped_existing"
                    else:
                        det_dst.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(saved_image_path, det_dst)
                        det_status = "copied_from_images"
                else:
                    det_status = save_page_resource(
                        resource, det_dst, overwrite=OVERWRITE_EXISTING_PAGES
                    )

                if page_action is None:
                    page_action = det_status

            row = {
                "library": library,
                "document": document,
                "page": page,
                "out_name": resource["out_name"],
                "source_kind": resource["kind"],
                "labels_status": labels_status,
                "ners_status": ners_status,
                "precheck_skipped_doc": False,
            }

            if page_action == "skipped_existing":
                skipped_rows.append(row)
                doc_skipped += 1
            else:
                copied_rows.append(row)
                doc_copied += 1

        processed_docs.append(
            {
                "library": library,
                "document": document,
                "requested_pages": len(requested_pages),
                "copied_pages": doc_copied,
                "skipped_existing_pages": doc_skipped,
                "missing_pages": doc_missing,
                "face_detections": 0 if not face_detections else len(face_detections),
                "ners": 0 if not ners else len(ners),
                "labels_status": labels_status,
                "ners_status": ners_status,
                "precheck_skipped_doc": False,
                "precheck_all_pages_present": precheck["all_pages_present"],
                "precheck_metadata_complete": precheck["metadata_complete"],
            }
        )

        """ print(
            "OK:",
            f"requested={len(requested_pages)}",
            f"copied={doc_copied}",
            f"skipped={doc_skipped}",
            f"missing={doc_missing}",
            f"face_detections={0 if not face_detections else len(face_detections)}",
            f"ners={0 if not ners else len(ners)}",
        ) """

    except Exception as e:
        failed_docs.append(
            {
                "library": library,
                "document": document,
                "error": repr(e),
            }
        )
        print("FAILED:", library, document, "->", repr(e))

In [ ]:
processed_docs_df = pd.DataFrame(processed_docs)
copied_df = pd.DataFrame(copied_rows)
skipped_df = pd.DataFrame(skipped_rows)
missing_df = pd.DataFrame(missing_rows)
failed_docs_df = pd.DataFrame(failed_docs)

precheck_skipped_docs = 0
if len(processed_docs_df) and "precheck_skipped_doc" in processed_docs_df.columns:
    precheck_skipped_docs = int(
        processed_docs_df["precheck_skipped_doc"].fillna(False).sum()
    )

print("\n=== Summary ===")
print("Processed docs              :", len(processed_docs_df))
print("Docs skipped by precheck    :", precheck_skipped_docs)
print("Copied pages                :", len(copied_df))
print("Skipped existing pages      :", len(skipped_df))
print("Missing pages               :", len(missing_df))
print("Failed docs                 :", len(failed_docs_df))

if len(failed_docs_df):
    display(failed_docs_df.head(20))

In [ ]:
selected_df.to_csv(EXPORT_ROOT / "selected_pages.csv", index=False)
selected_df.to_json(
    EXPORT_ROOT / "selected_pages.json", orient="records", force_ascii=False, indent=2
)

processed_docs_df.to_csv(EXPORT_ROOT / "processed_documents.csv", index=False)
copied_df.to_csv(EXPORT_ROOT / "copied_pages.csv", index=False)
skipped_df.to_csv(EXPORT_ROOT / "skipped_existing_pages.csv", index=False)
missing_df.to_csv(EXPORT_ROOT / "missing_pages.csv", index=False)
failed_docs_df.to_csv(EXPORT_ROOT / "failed_documents.csv", index=False)

print("Hotovo.")
print("Výstupný priečinok:", EXPORT_ROOT.resolve())